In [8]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import httpx
import openai_setup

endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
deployed_model = os.environ["DEPLOYMENT_NAME"]
print(f"Endpoint: {endpoint}. Model: {deployed_model}") 

token_provider = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default"
        )

token_provider()

In [ ]:

from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
import os

from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    SearchIndex,
    AzureOpenAIVectorizer,
    AzureOpenAIParameters
)





# The following variables from your .env file are used in this notebook
azure_search_endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
credential = AzureKeyCredential(os.getenv("AZURE_SEARCH_ADMIN_KEY", "")) if len(os.getenv("AZURE_SEARCH_ADMIN_KEY", "")) > 0 else DefaultAzureCredential()
index_name = "aml_index_2" #os.getenv("AZURE_SEARCH_INDEX", "vectest")
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.getenv("AZURE_OPENAI_KEY", "") if len(os.getenv("AZURE_OPENAI_KEY", "")) > 0 else None
azure_openai_deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o")
azure_openai_deployment_name_gpt_35_turbo = os.getenv("AZURE_OPENAI_ENDPOINT_GPT_35_Turbo", "gpt-4o")
azure_openai_deployment_name_gpt_35_turbo_key = os.getenv("AZURE_OPENAI_ENDPOINT_GPT_35_Turbo_API_KEY", "gpt-4o")
azure_openai_deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o")
azure_openai_embedding__large_deployment = os.getenv("AZURE_OPENAI_3_LARGE_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
azure_openai_embedding__small_deployment = os.getenv("AZURE_OPENAI_3_LARGE_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
azure_openai_embedding_large_dimensions = int(os.getenv("AZURE_OPENAI_EMBEDDING_LARGE_DIMENSIONS", 3072))
azure_openai_embedding_small_dimensions = int(os.getenv("AZURE_OPENAI_EMBEDDING_SMALLDIMENSIONS", 1536))
embedding_model_name = os.getenv("AZURE_OPENAI_3_LARGE_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-06-01")
azure_document_intelligence_endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT", "https://document-intelligence.api.cognitive.microsoft.com/")
azure_document_intelligence_key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY", "")

# print all the above
print(azure_search_endpoint)
print(credential)
print(index_name)
print(azure_openai_endpoint)

print(azure_openai_embedding__large_deployment)

print(azure_openai_embedding_large_dimensions)
print(azure_openai_embedding_small_dimensions)
print(embedding_model_name)
print(azure_openai_api_version)
print(azure_document_intelligence_endpoint)
print(azure_openai_deployment_name_gpt_35_turbo)
print(azure_openai_deployment_name_gpt_35_turbo_key)

In [18]:
from openai import AzureOpenAI
import openai


openai.api_key=azure_openai_deployment_name_gpt_35_turbo_key

client = AzureOpenAI(
   api_version="2024-05-01-preview",
   azure_deployment=deployed_model,
   azure_endpoint=endpoint,
   #azure_openai_key=azure_openai_deployment_name_gpt_35_turbo_key,
   #azure_ad_token_provider=token_provider,
)

def get_completion(
    messages: list[dict[str, str]],
    model: str = "gpt-4",
    max_tokens=500,
    temperature=0,
    stop=None,
    seed=123,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stop": stop,
        "seed": seed,
        "logprobs": logprobs,
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.chat.completions.create(**params)
    return completion

def getOpenAIResp(userQuery, systemMessage, deployed_model='gpt-4-turbo'):
    completion = client.chat.completions.create(
            model=deployed_model,
            messages=[
                {
                    "role": "system",
                    "content": systemMessage
                },
                {
                    "role": "user",
                    "content": userQuery
                }
            ],
            temperature=0,
            max_tokens=4000,
            stream=False)
    # print(completion.choices[0].message.content)
    return completion.choices[0].message.content

In [1]:
import pandas as pd


df = pd.read_json("data_asset_with_new_context_gpt-35-turbo.json", lines=True)

# create dataframe with columns question, context
qdf = pd.DataFrame(columns=['question', 'new_context'])
qdf['question'] = df['question']

In [12]:
new_data = pd.DataFrame([
    {'question': 'What are the key benefits of using Azure Machine Learning for managing and deploying machine learning models across different environments?'},
    {'question': 'How does Azure Machine Learning support automation in the machine learning lifecycle, including model training and hyperparameter tuning?'},
    {'question': 'Can you explain the role of Azure Machine Learning pipelines and how they help in orchestrating the end-to-end ML workflow?'},
    {'question': 'What security features are provided by Azure Machine Learning to ensure the protection of data and models?'},
    {'question': 'How does Azure Machine Learning integrate with other Azure services, such as Azure Data Lake and Azure Synapse Analytics, to enhance data management and processing for machine learning?'}
])

qdf = pd.concat([qdf, new_data], ignore_index=True)

In [ ]:
for i in range(len(qdf)):
    print(qdf['question'][i])

In [13]:
import pandas as pd
import numpy as np
from azure.search.documents.models import VectorizableTextQuery
from azure.search.documents.models import VectorFilterMode
from azure.search.documents.models import QueryType, QueryCaptionType, QueryAnswerType

#df = pd.read_json(data_asset.path, lines=True)


# append few rows to df


# Assuming df is already created



index_name_2 = "aml_index_with_suggester"
search_client = SearchClient(endpoint=azure_search_endpoint, index_name=index_name_2, credential=credential)

for i in range(qdf.shape[0]):
    query = qdf.iloc[i]["question"]    
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=10, fields="contentVector")
    results = search_client.search(  
    search_text=query,  
    vector_queries= [vector_query],
    #vector_filter_mode=VectorFilterMode.PRE_FILTER,
    #filter="category/any(c: c eq 'Networking')",
    select=["title", "content", "category", "tags", "docName"],
    #query_type=QueryType.SEMANTIC, semantic_configuration_name='aml-semantic-config', query_caption=QueryCaptionType.EXTRACTIVE, query_answer=QueryAnswerType.EXTRACTIVE,
    top=10
    )
    new_context = ""
    for result in results:  
        new_context += result['content'] + "\n\n"
        #df.at[i, "new_context"] = result['content']
        #print(f"Title: {result['title']}")  
        #print(f"Score: {result['@search.score']}")  
        #print(f"Reranker Score: {result['@search.reranker_score']}")
        #print(f"Content: {result['content']}")  
        #print(f"Category: {result['category']}")  
        #print(f"DocName: {result['docName']}\n\n")
    qdf.at[i, "new_context"] = new_context

qdf.to_json("data_asset_with_new_context_gpt-35-turbo.json", orient="records", lines=True)


In [19]:
def get_system_prompt_template(context, userQuery):
    systemPromptTemplate = f"""
    You need to respond whether you can answer the question with the retrived articles. 
    Before even answering the question, consider whether you have sufficient information in the article to answer the question fully.
    Your output should JUST be the boolean true or false, if you have sufficient information in the article to answer the question.
    Respond with just one word, the boolean true or false. You must output the word 'True', or the word 'False', nothing else.
    You must alway respond either True or False.   
    
    You retrieved this article: {context}. The question is: {userQuery}.
    
    Your answer: True or False
    """
    return systemPromptTemplate
# loop through the first 5 rows of the dataset
for i in range(qdf.shape[0]):
    
    #print(f"Question: {df['question'][i]}")
    #print(f"Context: {df['new_context'][i]}")
    messages = [
        { "role": "system", "content": get_system_prompt_template(qdf["new_context"][i], qdf["question"][i]) },
        { "role": "user", "content": "Can you answer the question or not?" }
    ]
    api_response = get_completion(messages, logprobs=True, top_logprobs=2, model=azure_openai_deployment_name_gpt_35_turbo)
    for logprob in api_response.choices[0].logprobs.content:
        # Dynamically create columns if they do not exist
        qdf.at[i, "has_sufficient_context_for_answer"] = logprob.token
        qdf.at[i, "logprobs"] = logprob.logprob
        qdf.at[i, "linear_probability"] = np.round(np.exp(logprob.logprob) * 100, 2)

    

#print(df.head()[0:5])

In [20]:
qdf.to_csv("self_eval_output_gpt-35-turbo.csv", index=False)